# Supertonic 3 — Voice Clone Studio (Google Colab GPU Workspace)

Web GUI untuk training dan speech generation Supertonic3 menggunakan GPU T4 gratis dari Google Colab.

### Langkah-langkah Penggunaan:
1. Hubungkan Runtime ke **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).
2. Unggah file `supertonic3.zip` ke Google Drive utama Anda.
3. Jalankan sel di bawah satu per satu untuk memulai setup dan server.
4. Klik link **Ngrok** yang muncul di akhir untuk masuk ke Web GUI.

In [ ]:
# ==========================================
# 1. MOUNT GOOGLE DRIVE & EKSTRAK ARCHIVE
# ==========================================
from google.colab import drive
import os

print(">> Menghubungkan ke Google Drive...")
drive.mount('/content/drive')

%cd /content
print(">> Mengekstrak file project supertonic3.zip...")
# Mengekstrak file zip kamu ke folder /content/project
!unzip -q /content/drive/MyDrive/supertonic3.zip -d /content/project

# Masuk ke folder root project
target_dir = "/content/project"
for root, dirs, files in os.walk("/content/project"):
    if "train_style.py" in files:
        target_dir = root
        break

%cd $target_dir
print(f">> Berada di folder kerja: {os.getcwd()}")

In [ ]:
# ==========================================
# 2. INSTALL DEPENDENCIES & SETUP GPU T4
# ==========================================
print(">> Menginstal pustaka dasar...")
!pip install -r requirements.txt

print(">> Memaksa instalasi PyTorch versi CUDA untuk GPU T4...")
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121 --force-reinstall

print(">> Membersihkan paket konflik (torchcodec)...")
!pip uninstall torchcodec -y

print(">> Menginstal FastAPI, Uvicorn, Python-Multipart, dan Pyngrok...")
!pip install fastapi uvicorn python-multipart pyngrok

In [ ]:
# ==========================================
# 3. PATCH AUTOMATIC BYPASS (SOUNDFILE)
# ==========================================
print(">> Memodifikasi utils/loss.py untuk menggunakan soundfile backend...")
loss_path = "utils/loss.py"

if os.path.exists(loss_path):
    with open(loss_path, "r") as f:
        code = f.read()

    new_func = """def load_audio_16khz_mono(file_path):
    import soundfile as sf
    data, sample_rate = sf.read(file_path)
    signal = torch.FloatTensor(data)
    if len(signal.shape) > 1:
        signal = torch.mean(signal, dim=1, keepdim=True)
    else:
        signal = signal.unsqueeze(0)
    if signal.shape[0] > signal.shape[1] and signal.shape[1] == 1:
        signal = signal.T
    if sample_rate != 16000:
        signal = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(signal)
    return signal"""

    lines = code.split('\n')
    new_lines = []
    skip = False
    for line in lines:
        if "def load_audio_16khz_mono" in line:
            skip = True
            new_lines.append(new_func)
            continue
        if skip and (line.startswith("class ") or line.startswith("import ") or line.startswith("from ")):
            skip = False
        if not skip:
            new_lines.append(line)

    with open(loss_path, "w") as f:
        f.write('\n'.join(new_lines))
    print(">> PADA KODE: utils/loss.py BERHASIL DI-PATCH!")
else:
    print(">> ERROR: File utils/loss.py tidak ditemukan!")

In [ ]:
# ==========================================
# 4. START NGROK TUNNEL & FASTAPI SERVER
# ==========================================
from pyngrok import ngrok

print(">> Mengatur token autentikasi Ngrok...")
ngrok.set_auth_token("3FUfeyBIqJutJdK5iZ9F9mlz9sI_7QjY5AcWcV6jkrKbXSEwE")

print(">> Membuka tunnel Ngrok di port 8000...")
try:
    ngrok.kill()
except:
    pass

public_url = ngrok.connect(8000)
print("\n" + "="*80)
print(f"🎉 WEB INTERFACE ACTIVE!")
print(f"👉 BUKA WEB UI SUPERTONIC DI SINI: {public_url}")
print("="*80 + "\n")

print(">> Menjalankan FastAPI Server...")
!python server.py